# Лабораторная работа 04. Визуализация данных, BI и Data Story

**Курс:** «Анализ данных и искусственный интеллект», 1 курс бакалавриата.

**Ситуация:** команда сервиса объявлений готовит инструмент оценки цены. Аналитику нужно показать состав имеющихся предложений, различия ценовых сегментов и проблемы данных, которые могут повлиять на будущую модель.

BI (Business Intelligence) здесь означает подготовку показателей и визуального отчёта для решения прикладной задачи. Мы строим такой отчёт в Python, без отдельной BI-платформы.

**Результат:** графики Matplotlib/Seaborn, интерактивное исследование Plotly, журнал проверяемых записей, компактная панель и Data Story с доказательствами и предлагаемым действием.

**Основной маршрут одной практики:** разделы 1–5 → по одному примеру из разделов 6–7 → панель в разделе 9 → Data Story в разделе 10 → отчёт в разделе 11. Остальные графики — каталог для самостоятельной работы. Виджеты и treemap — дополнительные задания; все примеры запускаются независимо от выполнения лабораторной 03 при наличии `cars_processed.csv`.


## 1. От бизнес-вопроса к показателю и графику

Заказчику нужен ответ, а не перечень методов библиотеки. Перед построением графика запишите вопрос, единицу наблюдения, показатель, сегмент и возможное действие.

| Вопрос | Показатель | Представление | Возможное действие |
|---|---|---|---|
| Какие марки лучше представлены? | Число и доля записей | Упорядоченные столбцы | Начать проверку модели с достаточно представленных сегментов |
| Какая цена типична для сегмента? | Медиана, Q1–Q3, число записей | Boxplot или столбцы с таблицей объёма | Выбрать ориентир для предварительной оценки |
| Отличается ли состав топлива по маркам? | Доли внутри марки | Тепловая карта процентов | Сравнивать сходные по составу сегменты |
| Какие записи требуют проверки? | Крайние значения и необычные сочетания | Гистограмма, scatter, интерактивная подсказка | Вернуться к строкам и проверить источник |

**Границы выводов:** число записей не равно спросу или скорости продаж; сумма цен не равна выручке; `Year` не является датой продажи. В выборке нет сведений о сделках, затратах, состоянии и комплектации. Отличия на графике дают основание для гипотезы, но не доказывают причинную связь.


## 2. Загрузка и подготовка

Положите `cars_processed.csv` рядом с ноутбуком. После лабораторной 03 можно использовать `cars_summary_input.csv`. Для диагностики необработанных данных дополнительно пригодится `cars.csv`. Цены — в евро, пробег — в километрах. Установка при необходимости: `pip install pandas numpy matplotlib seaborn plotly ipywidgets`. Перезапустите ядро после установки.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display

# Положите CSV рядом с ноутбуком. Для примера из репозитория — резервный адрес.
DATA_FILE = ('cars_summary_input.csv' if Path('cars_summary_input.csv').exists()
             else 'cars_processed.csv')
data_path = Path(DATA_FILE)
if data_path.exists():
    df = pd.read_csv(data_path)
else:
    df = pd.read_csv(
        'https://raw.githubusercontent.com/MVRonkin/BasicDataAnalysisCourse/'
        'refs/heads/main/Workshops/cars_processed.csv'
    )

# Совместимость с результатом предыдущей практики.
df = df.rename(columns={'Price(euro)': 'Price', 'Engine_capacity(cm3)': 'Engine'})
df = df.drop(columns=[c for c in df.columns if c.startswith('Unnamed:')])
if 'Transmission' not in df:
    df['Transmission'] = df['AutoTransmission'].map({True: 'Automatic', False: 'Manual'})
if 'km_year_cat' not in df:
    df['km_year_cat'] = pd.cut(
        df['km_year'], bins=[-np.inf, 10_000, 30_000, np.inf],
        labels=['Small', 'Medium', 'Large']
    )
# Номер строки нужен для возврата от графика к исходной записи; это не ID объявления.
if 'row_id' not in df:
    df['row_id'] = np.arange(len(df))
num_columns = ['Year', 'Distance', 'Engine', 'Price', 'km_year']
cat_columns = ['Make', 'Model', 'Style', 'Fuel_type', 'Transmission', 'km_year_cat']
df.head()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 100
df_base = df.copy()
print(f'Записей: {len(df):,}; марок: {df["Make"].nunique()}')

plt.show()


### Пять показателей для начала отчёта

KPI — ключевой показатель. В учебном отчёте это размер выборки, число марок и типичные цены/пробеги. Они характеризуют доступные данные; для KPI эффективности бизнеса потребовались бы сделки и затраты.


In [ ]:
kpi = pd.Series({
    'Записей, шт.': len(df),
    'Марок, шт.': df['Make'].nunique(),
    'Медиана цены, евро': df['Price'].median(),
    'IQR цены, евро': df['Price'].quantile(.75) - df['Price'].quantile(.25),
    'Медиана пробега, км': df['Distance'].median()
})
display(kpi.round(1).to_frame('Значение'))


## 3. Matplotlib: читаемый график для заказчика

`matplotlib.pyplot` (обычно `plt`) — интерфейс Matplotlib. `Figure` — весь рисунок; `Axes` — область одного графика. `fig, ax = plt.subplots()` позволяет отдельно настраивать рисунок и оси.

Сначала покажем обычную диаграмму количества записей по маркам, затем сделаем её удобнее: оставим топ-10, упорядочим значения, используем горизонтальные столбцы и добавим доли.


In [ ]:
# Создание фигуры 
plt.figure(figsize=(12, 4))

# Создание barchat
data = df['Make'].value_counts()

plt.bar(data.index, data.values)

# Добавляем заголовки и подписи
plt.title('Количество автомобилей по маркам', fontsize=14, pad=20)
plt.xlabel('Марка автомобиля', fontsize=12)
plt.ylabel('Количество', fontsize=12)

plt.grid(axis='y', alpha=0.25)

# делаем ось X красивой
plt.tick_params(axis='x', rotation=45, labelsize=10)


plt.show()


Если подписей слишком много, график плохо читается. Категории без естественного порядка удобно сортировать по показателю. Столбцы начинайте от нуля; цветом выделяйте смысл, а не каждую категорию ради украшения.


In [ ]:
counts = df['Make'].value_counts()
top = counts.head(10).sort_values()
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(top.index, top.values, color='#3973a5')
ax.bar_label(bars, labels=[f'{n:,} ({100*n/len(df):.1f}%)' for n in top], padding=4)
ax.set(title='Топ-10 марок по числу записей в выборке',
       xlabel='Число записей', ylabel='Марка')
ax.set_xlim(0, top.max() * 1.3)
ax.grid(axis='x', alpha=.25)
ax.grid(axis='y', visible=False)
fig.tight_layout()
plt.show()
print(f'Остальные марки: {len(df) - counts.head(10).sum():,} записей')


**Разбор:** проценты рассчитаны от всех записей, а не только от топ-10. График показывает охват выборки. Укажите три наиболее представленные марки и долю остальных; сформулируйте, какие сегменты разумно первыми использовать для проверки модели.

### Каталог: настройка цвета, сетки и подписей


In [ ]:
# Создаем фигуру и оси с указанием размера
plt.figure(figsize=(12, 6))

# Получаем количество автомобилей по типам топлива
# value_counts() автоматически сортирует по убыванию частоты
data = df['Fuel_type'].value_counts()

# Создаем столбчатую диаграмму с настройкой внешнего вида
bars = plt.bar(data.index, data.values, 
               color='lightgreen',      # Основной цвет столбцов
               edgecolor='darkgreen',   # Цвет границ столбцов
               alpha=0.7,               # Прозрачность (0-1)
               linewidth=1.5)           # Толщина границ

# НАСТРОЙКА ВНЕШНЕГО ВИДА ОСЕЙ:
# Поворачиваем подписи по оси X для лучшей читаемости
plt.tick_params(axis='x', rotation=45, labelsize=12)
# Настраиваем подписи по оси Y
plt.tick_params(axis='y', labelsize=12)

# ДОБАВЛЯЕМ ПОДПИСИ НАД СТОЛБЦАМИ:
# Проходим по каждому столбцу и добавляем текстовую метку
for bar in bars:
    height = bar.get_height()                    # Получаем высоту столбца
    plt.text(bar.get_x() + bar.get_width()/2.,   # Позиция X (центр столбца)
             height + 0.1,                       # Позиция Y (немного выше столбца)
             f'{int(height)}',                   # Текст (целое число)
             ha='center', va='bottom',           # Выравнивание по центру и снизу
             fontsize=11, fontweight='bold')     # Стиль шрифта

# ЗАГОЛОВКИ И ПОДПИСИ ОСЕЙ:
plt.title('Распределение автомобилей по типам топлива', 
          fontsize=16, pad=20, fontweight='bold')
plt.xlabel('Тип топлива', fontsize=14, labelpad=10)
plt.ylabel('Количество автомобилей', fontsize=14, labelpad=10)

# ДОБАВЛЯЕМ СЕТКУ ДЛЯ ЛУЧШЕЙ ЧИТАЕМОСТИ:
plt.grid(axis='y',    # Сетка только по вертикали (ось Y)
         alpha=0.3,   # Прозрачность сетки
         linestyle='--')  # Стиль линии (пунктир)

# АВТОМАТИЧЕСКИ НАСТРАИВАЕМ ОТСТУПЫ ДЛЯ КРАСИВОГО ОТОБРАЖЕНИЯ
plt.tight_layout()

# ОТОБРАЖАЕМ ГРАФИК
plt.show()

**Задание:** дайте диаграмме топлива содержательную подпись. Цвет должен быть читаемым и при печати. Подписи и порядок категорий должны совпадать на сравниваемых графиках.

## 4. Распределения: ищем проблемы данных

Гистограмма отвечает на вопрос, где сосредоточены значения и насколько длинный хвост распределения. `bins` задаёт интервалы, а не категории. Число интервалов может скрыть или подчеркнуть детали; сравните несколько вариантов. Обрабатываемые данные уже прошли первичную очистку, поэтому самые грубые ошибки могли исчезнуть до этой работы.


In [ ]:
# Создаем фигуру с несколькими subplots
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle('Анализ распределения ключевых показателей автомобилей', 
             fontsize=16, fontweight='bold', y=0.97)

# Гистограмма цен
axes[0,0].hist(df['Price'], bins=50, color='skyblue', edgecolor='navy', alpha=0.7)
axes[0,0].set_title('Распределение цен')
axes[0,0].set_xlabel('Цена, евро')
axes[0,0].set_ylabel('Количество')
axes[0,0].grid(alpha=0.3)

# Гистограмма годов
axes[0,1].hist(df['Year'], bins=range(1990, 2025, 2), color='lightgreen', 
               edgecolor='darkgreen', alpha=0.7)
axes[0,1].set_title('Распределение по годам выпуска')
axes[0,1].set_xlabel('Год выпуска')
axes[0,1].set_ylabel('Количество')
axes[0,1].grid(alpha=0.3)

# Гистограмма пробега в год
km_filtered = df['km_year']
axes[1,0].hist(km_filtered, bins=20, color='lightcoral', 
               edgecolor='darkred', alpha=0.7)
axes[1,0].set_title('Распределение годового пробега')
axes[1,0].set_xlabel('Пробег за год (км)')
axes[1,0].set_ylabel('Количество')
axes[1,0].grid(alpha=0.3)

# Гистограмма зависимости цены от пробега (hexbin)
km_ = df['Distance']
axes[1,1].hist(km_, bins=20, color='lightcoral', 
               edgecolor='darkred', alpha=0.7)
axes[1,1].set_title('Распределение  пробега')
axes[1,1].set_xlabel('Пробег (км)')
axes[1,1].set_ylabel('Количество')
axes[1,1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

**Разбор:** сравните центр и хвосты четырёх распределений. Запишите, какой диапазон представлен лучше и какие значения нужно проверить. `km_year` рассчитан из пробега и возраста; это условный среднегодовой пробег, а не реальные измерения по каждому году эксплуатации.

### Масштаб и число интервалов


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(df['Price'].dropna(), bins=20)
axes[0].set(title='Цена: 20 интервалов', xlabel='Цена, евро', ylabel='Число записей')
axes[1].hist(df['Price'].dropna(), bins=80)
axes[1].set(title='Цена: 80 интервалов', xlabel='Цена, евро')
positive = df.loc[df['Price'] > 0, 'Price']
log_bins = np.geomspace(positive.min(), positive.max(), 40)
axes[2].hist(positive, bins=log_bins)
axes[2].set_xscale('log')
axes[2].set(title='Цена: логарифмическая ось', xlabel='Цена, евро (log)')
fig.tight_layout()
plt.show()
print('Неположительных или отсутствующих цен:', int((~(df['Price'] > 0)).sum()))


На логарифмической оси одинаковый шаг соответствует одинаковому отношению цен. Единицы остаются евро. Такой масштаб помогает увидеть широкий диапазон; он не исправляет значения и не даёт основания их удалить.


### Seaborn: короткая запись статистических графиков


In [ ]:
sns.histplot(data=df, x='Year', bins = 20);
plt.show()


In [ ]:
sns.histplot(df['km_year'], bins = 50, kde=True);
plt.show()


In [ ]:
sns.histplot(data=df, x='Distance', hue='Transmission', multiple='stack', bins = 20);
plt.show()


Суммарная гистограмма показывает и размер групп, и форму распределений. Если групп разное число, большая группа может доминировать. Для сравнения формы можно нормировать каждую группу отдельно.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
sns.histplot(data=df, x='Distance', hue='Transmission', bins=40,
             stat='density', common_norm=False, element='step', fill=False, ax=ax)
ax.set(title='Форма распределения пробега по трансмиссии',
       xlabel='Пробег, км', ylabel='Плотность')
plt.show()


### Boxplot: медиана, разброс и кандидаты на проверку

Коробка показывает Q1–Q3, линия внутри — медиану. IQR = Q3 − Q1. При стандартном `whis=1.5` усы доходят до крайних наблюдений в пределах [Q1 − 1.5·IQR, Q3 + 1.5·IQR]. Точки снаружи — статистически крайние значения; они могут быть корректными, например для редкого дорогого автомобиля.


In [ ]:
# Создаём фигуру с двумя подграфиками (1 строка, 2 столбца)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Гистограмма
sns.histplot(data=df, x='Year', bins=20, kde=False, ax=axes[0], color='skyblue', edgecolor='black')
axes[0].set_title('Распределение годов (гистограмма)')
axes[0].set_xlabel('Год')
axes[0].set_ylabel('Частота')

# Boxplot
sns.boxplot(data=df, y='Year', ax=axes[1], color='lightgreen')
axes[1].set_title('Распределение годов (boxplot)')
axes[1].set_ylabel('Год')
axes[1].set_xlabel('')

# Улучшаем отображение
plt.tight_layout()
plt.show()

**Разбор:** найдите годы выпуска за пределами усов. Проверьте несколько записей по марке, модели, цене и пробегу. Не путайте старый автомобиль с ошибочным годом.

Посмотрим на все численные признаки. Число областей автоматически соответствует числу колонок.


In [ ]:
fig, axes = plt.subplots(1, len(num_columns), figsize=(3 * len(num_columns), 4))
for ax, col in zip(axes, num_columns):
    sns.boxplot(data=df, y=col, ax=ax, color='lightblue')
    ax.set_title(col)
fig.tight_layout()
plt.show()


### Дополнительно: что было скрыто первичной очисткой

Пропуски, точные повторы и крайние значения исследуются на исходном `cars.csv`. Для сравнения версий используйте одинаковые единицы и масштабы. Изменение распределения после очистки может менять и последующие выводы.


In [ ]:
raw_path = Path('cars.csv')
if raw_path.exists():
    raw = pd.read_csv(raw_path).rename(columns={'Price(euro)': 'Price'})
    missing = raw.isna().mean().mul(100).sort_values(ascending=False)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].barh(missing.index, missing.values)
    axes[0].set(title='Пропуски в исходных данных', xlabel='Доля, %')
    for table, label in [(raw, 'Исходные'), (df, 'Подготовленные')]:
        p = table.loc[table['Price'] > 0, 'Price']
        axes[1].hist(np.log10(p), bins=50, density=True, histtype='step', label=label)
    axes[1].set(title='Цена до и после первичной очистки',
                xlabel='log10(цены в евро)', ylabel='Плотность')
    axes[1].legend()
    fig.tight_layout()
    plt.show()
    print('Точных повторов исходных строк:', int(raw.duplicated().sum()))
else:
    print('Положите cars.csv рядом с ноутбуком для сравнения с исходными данными.')


## 5. Совместное поведение признаков и журнал проверки

Scatter показывает сочетания значений. При наложении точек используйте прозрачность, меньший размер маркера или `hexbin`. Цвет применяйте к небольшому числу категорий. Размер как ещё один канал может перегрузить график.


In [ ]:
plt.figure(figsize=(12,5))
sns.scatterplot(    
    data=df,
    x="Distance", 
    y="Price",
    hue="Transmission", 
    size = "km_year", 
    alpha = 0.7,
    palette='bwr'
    );
plt.show()


**Разбор:** найдите необычные сочетания цены и пробега, затем откройте соответствующие строки. Высокая цена у автомобиля с большим пробегом может объясняться моделью и комплектацией. Общая разница цен по трансмиссии ещё не доказывает влияние трансмиссии: группы могут различаться по маркам и годам выпуска.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
hb = ax.hexbin(df['Distance'], df['Price'], gridsize=45, bins='log', mincnt=1)
fig.colorbar(hb, ax=ax, label='Число записей в ячейке (log)')
ax.set(title='Где сосредоточены сочетания цены и пробега',
       xlabel='Пробег, км', ylabel='Цена, евро')
plt.show()


### От графика к проверяемым строкам

Сохраняем флаги и причину проверки, не изменяя исходные значения. Пороги ниже взяты из прежнего учебного примера: они выделяют кандидатов, но не являются доказанными границами корректности.

Для каждой записи задайте вопрос: ошибка единиц? ошибка ввода? редкий, но допустимый объект? недостаточно сведений?


In [ ]:
checks = pd.DataFrame(index=df.index)
checks['Старый год выпуска'] = df['Year'] < 1971
checks['Большой пробег'] = df['Distance'] >= 800_000
checks['Высокая цена'] = df['Price'] >= 120_000
checks['Большой условный годовой пробег'] = df['km_year'] >= 80_000
checks['Высокая цена при большом пробеге'] = (df['Price'] >= 40_000) & (df['Distance'] >= 300_000)
checks['Неположительная цена'] = df['Price'].isna() | (df['Price'] <= 0)
checks['Отрицательный пробег'] = df['Distance'].isna() | (df['Distance'] < 0)
flagged = checks.any(axis=1)
review = df.loc[flagged, ['row_id', 'Make', 'Model', 'Year', 'Distance', 'Price']].copy()
review['Причина проверки'] = checks.loc[flagged].apply(
    lambda row: '; '.join(row.index[row]), axis=1
)
review['Решение'] = 'Требует проверки'
display(review.head(20))
display(checks.sum().sort_values(ascending=False).to_frame('Записей'))
print(f'Всего кандидатов: {flagged.sum()} ({flagged.mean():.2%})')


### Сценарий чувствительности отчёта к фильтрации

Сравним полный набор с **сценарием** исключения всех кандидатов. Такая фильтрация не считается подтверждённой очисткой. Она показывает, насколько выбранные правила способны изменить показатели.


In [ ]:
scenario = df.loc[~flagged].copy()
comparison = pd.DataFrame({
    'Полный набор': [len(df), df['Price'].mean(), df['Price'].median()],
    'Сценарий без кандидатов': [len(scenario), scenario['Price'].mean(), scenario['Price'].median()]
}, index=['Число записей', 'Средняя цена, евро', 'Медиана цены, евро'])
display(comparison.round(1))
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharex=True, sharey=True)
for ax, table, title in zip(axes, [df, scenario], ['Полный набор', 'Сценарий без кандидатов']):
    sns.scatterplot(data=table, x='Distance', y='Price', s=10, alpha=.2, ax=ax)
    ax.set(title=title, xlabel='Пробег, км', ylabel='Цена, евро')
fig.tight_layout()
plt.show()


**Задание:** проверьте пять кандидатов и заполните решения с обоснованием. Не удаляйте запись, если сведений недостаточно. Сравните число записей, среднюю и медианную цену до и после **обоснованного** решения. Какие сегменты теряются?

## 6. Сравнение категорий и сегментов

Рейтинг по средней цене способен скрыть разброс и маленькие группы. Сначала смотрим число объектов, затем типичную цену и диапазон. Ниже — несколько способов показать различия.


In [ ]:
segment = df.groupby('Make').agg(n=('Price', 'size'),
    mean_price=('Price', 'mean'), median_price=('Price', 'median'))
selected_makes = segment.nlargest(10, 'n').index
plot_table = segment.loc[selected_makes].sort_values('median_price')
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(plot_table.index, plot_table['median_price'], color='#3973a5')
ax.set(title='Типичная цена хорошо представленных марок',
       xlabel='Медианная цена, евро', ylabel='Марка')
for y, (_, row) in enumerate(plot_table.iterrows()):
    ax.text(row['median_price'], y, f'  n={int(row["n"]):,}', va='center')
ax.set_xlim(0, plot_table['median_price'].max() * 1.3)
plt.show()
display(plot_table.round(1))


Медиана показывает типичный уровень, но не ширину ценового сегмента. Покажем распределения по кузову и условному годовому пробегу.


In [ ]:
plt.figure(figsize= (12,3))

sns.boxplot(data=df, x='Style', y='Price')

plt.grid(axis='y', alpha=0.25)
plt.xticks(rotation=45, ha='right')  # ha='right' — чтобы подписи не "уезжали"
plt.title('Распределение цен по типу кузова')
plt.tight_layout()  # важно: предотвращает обрезание подписей
plt.show()

In [ ]:
plt.figure(figsize= (12,3))

sns.boxplot(data=df, x='km_year_cat', y='Price');
plt.grid(axis='y', alpha=0.25)
plt.xticks(rotation=45, ha='right')  # ha='right' — чтобы подписи не "уезжали"
plt.title('Цена в зависимости от категории годового пробега')
plt.tight_layout()  # важно: предотвращает обрезание подписей
plt.show()



**Разбор:** сравните медиану, IQR и число записей в двух выбранных группах. Широкая коробка означает неоднородность цен, а не автоматически плохие данные. Наличие дорогих точек не доказывает завышение цены.

### Малые кратные: одинаковый график для разных марок

`catplot` строит сетку панелей. Общая шкала помогает сравнить группы, но различия моделей и возраста всё ещё могут влиять на результат.


In [ ]:
# Берём топ-6 брендов
top_makes = df['Make'].value_counts().head(6).index
df_top = df[df['Make'].isin(top_makes)]

sns.catplot(
    order=['Small', 'Medium', 'Large'],
    x='km_year_cat',
    y='Price',
    col='Make',
    data=df_top.dropna(subset=['km_year_cat', 'Price']),
    kind='box',
    col_wrap=3,
    height=4,
    aspect=0.9
)
plt.tight_layout()
plt.show()

### Уточнение сравнения: одна модель и годы выпуска

Сравним трансмиссии для одной достаточно представленной модели в одном пятилетнем диапазоне. Это уменьшает различия состава, но не устраняет все факторы: состояние и комплектация неизвестны.


In [ ]:
detail = df.copy()
detail['year_band'] = (detail['Year'] // 5) * 5
counts_detail = detail.groupby(['Make', 'Model', 'year_band', 'Transmission']).size().unstack(fill_value=0)
if {'Automatic', 'Manual'}.issubset(counts_detail.columns):
    eligible = counts_detail[(counts_detail['Automatic'] >= 15) & (counts_detail['Manual'] >= 15)]
else:
    eligible = pd.DataFrame()
if not eligible.empty:
    chosen = eligible.sum(axis=1).idxmax()
    make, model, band = chosen
    same_segment = detail[(detail['Make'] == make) & (detail['Model'] == model) & (detail['year_band'] == band)]
    fig, ax = plt.subplots(figsize=(8, 4))
    sns.boxplot(data=same_segment, x='Transmission', y='Price', ax=ax)
    ax.set(title=f'{make} {model}, годы выпуска {band}–{band + 4}',
           xlabel='Трансмиссия', ylabel='Цена, евро')
    plt.show()
    display(same_segment.groupby('Transmission')['Price'].agg(['size', 'median', 'mean']).round(1))
else:
    print('Нет сопоставимого сегмента с хотя бы 15 записями каждого типа трансмиссии.')


**Задание:** сравните общий график по трансмиссиям с графиком внутри сегмента. Насколько изменился разрыв? Сформулируйте вывод без утверждения причинного влияния.

### Связь с лабораторной 03: тепловые карты сводных таблиц


In [ ]:
fuel_by_make = pd.crosstab(df['Make'], df['Fuel_type'])

fuel_by_make = fuel_by_make.loc[fuel_by_make.sum(axis=1).nlargest(10).index]  # топ-10 брендов

sns.heatmap(fuel_by_make, annot=True, fmt='d', cmap='Blues')
plt.title('Топ-10 брендов по типу топлива')
plt.xticks(rotation=45, ha='right')  # ha='right' — чтобы подписи не "уезжали"
plt.tight_layout()  # важно: предотвращает обрезание подписей
plt.show()

Абсолютные числа выделяют марки с большим объёмом выборки. Доли отвечают на вопрос о составе предложений внутри каждой марки. Покажем обе таблицы с одинаковым порядком категорий.


In [ ]:
fuel_share = fuel_by_make.div(fuel_by_make.sum(axis=1), axis=0) * 100
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.heatmap(fuel_by_make, annot=True, fmt='d', cmap='Blues', ax=axes[0])
sns.heatmap(fuel_share, annot=True, fmt='.1f', cmap='Blues', vmin=0, vmax=100, ax=axes[1])
axes[0].set_title('Число записей')
axes[1].set_title('Структура внутри марки, %')
for ax in axes:
    ax.set(xlabel='Тип топлива', ylabel='Марка')
    ax.tick_params(axis='x', rotation=45)
fig.tight_layout()
plt.show()


**Разбор:** найдите пример, где большой абсолютный счётчик не соответствует большой доле. Проценты строки суммируются до 100%. Ноль в таблице количества означает отсутствие записей; отсутствие цены в ценовой таблице должно оставаться пропуском.

**Дополнительное задание:** проверьте, оправдано ли объединение `Electric`, `Hybrid` и `Plug-in Hybrid` для конкретного отчёта. Это разные типы силовой установки; не объединяйте их только из-за малого числа записей.

## 7. Plotly: от точки к исходной записи

Интерактивная визуализация позволяет увеличивать участок, отключать категории и читать параметры точки при наведении. У каждой записи показываем `row_id`, чтобы вернуться к таблице. Для читаемости цветом выделяем топ-6 марок; остальные группируем только в отображении.


In [ ]:
df_viz = df.copy()
top6 = df['Make'].value_counts().head(6).index
df_viz['Марка на графике'] = df_viz['Make'].where(df_viz['Make'].isin(top6), 'Другие')
fig = px.scatter(df_viz, x='Year', y='Price', color='Марка на графике',
    hover_data=['row_id', 'Make', 'Model', 'Distance', 'Fuel_type', 'Transmission'],
    opacity=.45, title='Цена предложения и год выпуска',
    labels={'Price': 'Цена, евро', 'Year': 'Год выпуска'})
fig.update_layout(height=550)
fig.show()


**Задание:** увеличьте область с подозрительными точками. Выберите три `row_id`, найдите строки в `df` и дополните журнал проверки. Учитывайте, что цвет «Другие» объединяет марки только на графике.


In [ ]:
df_with_index = df.copy()
df_with_index['index_'] = df_with_index.index

fig = px.box(
    df_with_index,
    x='Fuel_type',
    y='Price',
    color='Transmission',
    hover_data=['row_id', 'Make', 'Model'],  # ← индекс и другая полезная инфа
    title='Цена по типу топлива и коробке передач (наведите на выбросы)',
    labels={'Price': 'Цена, евро', 'Fuel_type': 'Тип топлива'}
)
fig.update_layout(height=500)
fig.show()

Boxplot в Plotly помогает изучить крайние значения по типу топлива и трансмиссии. Критерий усов остаётся статистическим; стоимость сделки по этой диаграмме определить нельзя.


In [ ]:
# Добавляем индекс как колонку
df_with_index = df.copy()
df_with_index['ID'] = df_with_index.index

fig = px.histogram(
    df_with_index,
    x='Price',
    color='km_year_cat',
    marginal='rug',  # ← rug показывает каждую точку
    hover_data=['row_id', 'Make', 'Model'],  # ← индекс и другая инфа
    opacity=0.7,
    nbins=50,
    title='Распределение цен по категориям годового пробега (наведите на точки внизу)'
)
fig.update_layout(bargap=0.1)
fig.show()

Гистограмма показывает количество записей в интервалах; отдельная запись доступна в нижней полосе точек (`rug`). Подумайте, что добавляет интерактивность и какие выводы можно оставить на статическом рисунке.


## 8. Дополнительно: treemap и фильтры

Treemap показывает иерархию «марка → модель»: площадь — число записей, цвет — средняя цена. Она удобна для обзора состава, но точные различия величин проще читать по столбцам. Среднее на уровне марки агрегирует представленные модели; разные цены моделей могут скрываться внутри одной марки.


In [ ]:
model_stats = df.groupby(['Make', 'Model']).agg(
    avg_price=('Price', 'mean'),
    count=('Price', 'size')
).reset_index()

fig = px.treemap(
    model_stats,
    path=['Make', 'Model'],
    values='count',
    color='avg_price',
    color_continuous_scale='RdYlBu_r',
    title='Дерево брендов и моделей (размер — количество, цвет — средняя цена)'
)
fig.show()

### Фильтры для исследования конкретного сегмента

Виджеты работают в среде с поддержкой `ipywidgets` (например, JupyterLab). Для выбранного сегмента показываем объём выборки, медианную цену и точки. Очень узкие фильтры могут оставить мало наблюдений; это ограничение надо сообщать пользователю. Для просмотра экспортированного HTML ядро Python и виджеты не требуются — он создаётся отдельно в следующем разделе.


In [ ]:
import pandas as pd
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display, clear_output

# === 1. Подготовка данных ===
df_viz = df.copy()
df_viz['ID'] = df_viz['row_id']  # добавляем ID

# Уникальные категории для фильтров
makes = ['(Все)'] + sorted(df_viz['Make'].dropna().unique().tolist())
fuel_types = ['(Все)'] + sorted(df_viz['Fuel_type'].dropna().unique().tolist())
transmissions = ['(Все)'] + sorted(df_viz['Transmission'].dropna().unique().tolist())
km_cats = ['(Все)'] + sorted(df_viz['km_year_cat'].dropna().unique().tolist())

# === 2. Виджеты фильтров ===
make_widget = widgets.Dropdown(options=makes, value='(Все)', description='Бренд:')
fuel_widget = widgets.Dropdown(options=fuel_types, value='(Все)', description='Топливо:')
trans_widget = widgets.Dropdown(options=transmissions, value='(Все)', description='Коробка:')
km_widget = widgets.Dropdown(options=km_cats, value='(Все)', description='Пробег/год:')

# === 3. Функция обновления графика и таблицы ===
output = widgets.Output()

def update_plot(change=None):
    # Фильтрация
    filtered = df_viz.copy()
    
    if make_widget.value != '(Все)':
        filtered = filtered[filtered['Make'] == make_widget.value]
    if fuel_widget.value != '(Все)':
        filtered = filtered[filtered['Fuel_type'] == fuel_widget.value]
    if trans_widget.value != '(Все)':
        filtered = filtered[filtered['Transmission'] == trans_widget.value]
    if km_widget.value != '(Все)':
        filtered = filtered[filtered['km_year_cat'] == km_widget.value]
    
    # Очистка вывода
    with output:
        clear_output(wait=True)
        
        if filtered.empty:
            print("Нет данных по выбранным фильтрам.")
            return
        
        # Построение графика
        fig = px.scatter(
            filtered,
            x='Year',
            y='Price',
            color='Make',
            hover_data=['row_id', 'Model', 'Distance', 'Engine', 'Fuel_type', 'Transmission'],
            title=f'Записей: {len(filtered)}; медиана цены: {filtered["Price"].median():,.0f} евро',
            labels={'Price': 'Цена, евро', 'Year': 'Год выпуска'}
        )
        fig.update_layout(height=500, showlegend=False)
        fig.show()
        
        # Вывод таблицы (первые 10 записей)
        display(filtered[['row_id', 'Make', 'Model', 'Year', 'Price', 'Distance']].head(10))

# === 4. Привязка событий и запуск ===
make_widget.observe(update_plot, names='value')
fuel_widget.observe(update_plot, names='value')
trans_widget.observe(update_plot, names='value')
km_widget.observe(update_plot, names='value')

# Отображение интерфейса
controls = widgets.VBox([
    widgets.HTML("<h3>Фильтры для анализа автомобилей</h3>"),
    widgets.HBox([make_widget, fuel_widget]),
    widgets.HBox([trans_widget, km_widget])
])

display(controls)
update_plot()  # первый запуск
display(output)

## 9. Компактная BI-панель

Панель отвечает на несколько связанных вопросов на одном экране: сколько данных доступно, какие марки представлены и как распределены цены. Сохраняйте одинаковые единицы, минимум декоративных элементов и понятные заголовки.

Ниже показан полный набор. Для панели выбранного сегмента сначала отфильтруйте `dashboard_df`; тогда **все** показатели и графики должны пересчитываться по нему. Укажите активный фильтр в заголовке.


In [ ]:
dashboard_df = df.copy()
DASHBOARD_FILTER = 'Все записи'
# Пример сегмента (раскомментируйте обе строки):
# dashboard_df = df[df['Transmission'] == 'Automatic'].copy()
# DASHBOARD_FILTER = 'Только Automatic'
if dashboard_df.empty:
    raise ValueError('Выбранный сегмент не содержит записей')

fig = make_subplots(rows=2, cols=3,
    specs=[[{'type': 'indicator'}] * 3,
           [{'type': 'xy'}, {'type': 'xy'}, {'type': 'xy'}]],
    subplot_titles=['', '', '', 'Топ-8 марок', 'Распределение цены', 'Цена по трансмиссии'],
    vertical_spacing=.2)
metrics = [('Записей, шт.', len(dashboard_df)),
           ('Медиана цены, евро', dashboard_df['Price'].median()),
           ('Медиана пробега, км', dashboard_df['Distance'].median())]
for column, (label, value) in enumerate(metrics, start=1):
    fig.add_trace(go.Indicator(mode='number', value=float(value),
        title={'text': label}, number={'valueformat': ',.0f'}), row=1, col=column)
brand_counts = dashboard_df['Make'].value_counts().head(8).sort_values()
fig.add_trace(go.Bar(x=brand_counts.values, y=brand_counts.index, orientation='h',
    marker_color='#3973a5'), row=2, col=1)
fig.add_trace(go.Histogram(x=dashboard_df['Price'], nbinsx=50, marker_color='#3973a5'), row=2, col=2)
for transmission, group in dashboard_df.groupby('Transmission'):
    fig.add_trace(go.Box(y=group['Price'], name=str(transmission), boxpoints=False), row=2, col=3)
fig.update_xaxes(title_text='Число записей', row=2, col=1)
fig.update_xaxes(title_text='Цена, евро', row=2, col=2)
fig.update_yaxes(title_text='Число записей', row=2, col=2)
fig.update_yaxes(title_text='Цена, евро', row=2, col=3)
fig.update_layout(height=720, showlegend=False,
    title=f'Предложения автомобилей — {DASHBOARD_FILTER}', template='plotly_white')
fig.show()
fig.write_html('cars_bi_dashboard.html', include_plotlyjs=True, full_html=True)


**Разбор:** панель сохраняется в `cars_bi_dashboard.html` и открывается в браузере без Python и сети. Графики интерактивны, но изменение фильтра в коде требует повторного выполнения ячейки. Boxplot в панели скрывает отдельные крайние точки ради компактности; это нужно учитывать при просмотре панели и проверять их в отдельном графике.

## 10. Data Story: объясняем результат и предлагаем следующий шаг

**Data Story** — последовательный рассказ, который связывает вопрос аудитории, наблюдения в данных, их интерпретацию и предлагаемое действие. Графики служат доказательствами сформулированных наблюдений.

**Цель этого раздела:** подготовить короткое сообщение для руководителя команды сервиса объявлений. Ему нужно решить, с каких сегментов начать проверку инструмента оценки цены и какие ограничения сообщить пользователю.

BI-панель помогает исследовать несколько показателей и менять взгляд на данные. В истории аналитик выбирает один вопрос и порядок доказательств. Панель может быть приложением к истории. Сам рассказ должен быть понятен без устного пояснения автора.

Мы пока не проверяли точность модели. Поэтому история может предложить план пилота, но не обещать качество будущей оценки.

### 10.1. Сначала заполняем паспорт истории

| Элемент | Пример для этой работы |
|---|---|
| Аудитория | Руководитель команды сервиса объявлений |
| Решение | Какие сегменты включить в первую проверку инструмента? |
| Доступные данные | Подготовленная таблица автомобилей, цена предложения и характеристики |
| Главный вопрос | Позволяет ли объём данных обещать одинаковый охват всех сегментов? |
| Нужные доказательства | Доли марок, различия цен, число записей внутри марок и моделей |
| Ограничения | Нет фактических сделок, ошибок модели и независимой проверки её качества |
| Возможное действие | Выбрать кандидатов для пилота, оценить ошибки по сегментам и отдельно обозначить малые группы |

**Упражнение до построения графиков:** одним предложением объясните, кому вы пишете и какое решение должны поддержать. Уберите графики, которые не помогают ответить на этот вопрос.

### 10.2. Каркас рассказа

1. **Контекст:** что команда собирается сделать и почему возник вопрос.
2. **Наблюдение:** что непосредственно посчитано или показано на графике.
3. **Интерпретация:** что это означает для поставленного вопроса; какие альтернативные объяснения возможны.
4. **Действие:** что предлагается проверить, изменить или собрать дальше.
5. **Ограничение:** какой вывод имеющиеся данные не подтверждают.

На одну историю достаточно **одной главной мысли и 2–3 доказательств**. Не нужно перечислять все диаграммы из каталога. Сначала исследуйте данные, затем выбирайте рассказ. Если расчёты противоречат первоначальной гипотезе, меняйте вывод, а не скрывайте неудобные значения.

Пример допустимой главной мысли: «Данные распределены по сегментам неравномерно; охват пилота стоит выбирать и оценивать по сегментам». Конкретные числа ниже берём из текущей таблицы.

### 10.3. Фиксируем набор и условия расчётов

Все три графика истории должны использовать один описанный набор. Ниже оставляем записи с известной маркой и моделью и конечной положительной ценой. Покажем число исключённых строк. Это условие учебного отчёта, а не новая подтверждённая очистка.

Для примера возьмём топ-5 марок. Величину «не менее 30 записей на марку–модель» используем как учебный порог для планирования проверки. **30 не является доказанным минимумом для точного прогноза или статистической гарантией.** Порог проверим на чувствительность.

Важно: `Rare` и `Other` могут объединять несколько исходных марок. Это категории после преобразования, а не реальные марки. Если есть `Make_original`, восстановим исходные марки в отдельной копии для истории. Если их нет, будем говорить о категориях поля Make и обозначим ограничение детализации. «Другие марки» на первом графике объединяет остаток только для отображения.

In [ ]:
story_make = (df['Make_original'].fillna(df['Make']) if 'Make_original' in df else df['Make'])
story_aggregate = story_make.isin(['Rare', 'Other', 'Другие']).any()
STORY_MAKE_WORD = 'категорий поля Make' if story_aggregate else 'марок'
STORY_SEGMENT_NAME = 'категория Make–модель' if story_aggregate else 'марка–модель'
print('Поле марки для истории:', 'Make_original' if 'Make_original' in df else 'Make')
if story_aggregate:
    print('Есть объединённые категории: детализация по исходным маркам ограничена.')
story_valid = (
    story_make.notna() & df['Model'].notna()
    & np.isfinite(df['Price']) & (df['Price'] > 0)
)
story_df = df.loc[story_valid].copy()
story_df['Make'] = story_make.loc[story_valid]
if story_df.empty:
    raise ValueError('Нет записей для выбранного определения отчёта.')
STORY_TOP_K = min(5, story_df['Make'].nunique())
STORY_MIN_N = 30
story_total = len(story_df)
story_counts = story_df['Make'].value_counts()
story_top_makes = story_counts.head(STORY_TOP_K).index
story_top_n = int(story_counts.head(STORY_TOP_K).sum())
story_top_share = 100 * story_top_n / story_total
story_model_n = story_df.groupby(['Make', 'Model'])['Price'].transform('size')
story_small_mask = story_model_n < STORY_MIN_N
story_small_n = int(story_small_mask.sum())
story_small_share = 100 * story_small_n / story_total

print('Источник:', DATA_FILE if data_path.exists() else 'Резервный CSV из репозитория')
print('В исходном df:', len(df), 'в истории:', story_total,
      'исключено по условиям истории:', int((~story_valid).sum()))
print('Цена: евро; наблюдение: запись, а не подтверждённая сделка.')
print(f'Топ-{STORY_TOP_K} {STORY_MAKE_WORD}: {story_top_n:,} записей ({story_top_share:.1f}%).')
print(f'В группах марка–модель с n < {STORY_MIN_N}: {story_small_n:,} записей ({story_small_share:.1f}%).')
story_dir = Path('cars_data_story_assets')
story_dir.mkdir(exist_ok=True)

### 10.4. График 1 — где сосредоточены записи

Заголовок сообщает проверяемое наблюдение. Остальные марки показаны одной дополнительной категорией: их доля не исчезает. Проценты считаются от всех записей story_df. Цвет выделяет кандидатов для изучения; он не означает успешные продажи или высокую точность.

In [ ]:
story_coverage = story_counts.head(STORY_TOP_K).copy()
remaining_n = story_total - int(story_coverage.sum())
if remaining_n:
    story_coverage = pd.concat([story_coverage, pd.Series({'Другие марки': remaining_n})])
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#287a9e' if name in story_top_makes else '#b7c0c8' for name in story_coverage.index]
bars = ax.barh(story_coverage.index, story_coverage.values, color=colors)
ax.invert_yaxis()
ax.bar_label(bars, labels=[f'{value:,} · {100*value/story_total:.1f}%' for value in story_coverage], padding=5)
ax.set_title(f'На топ-{STORY_TOP_K} {STORY_MAKE_WORD} приходится {story_top_share:.1f}% записей', loc='left', pad=18)
ax.set_xlabel('Число записей в подготовленном наборе')
ax.set_xlim(0, story_coverage.max() * 1.38)
ax.grid(axis='y', visible=False)
fig.text(.02, .01, f'База: {story_total:,} записей; известны марка и модель, цена конечна и > 0. Доли от всей базы.', fontsize=9)
fig.tight_layout(rect=(0, .06, 1, 1))
fig.savefig(story_dir / '01_coverage.png', dpi=140, bbox_inches='tight')
plt.show()

**Наблюдение:** назовите топ-5 и их суммарную долю, как она рассчитана в текущем запуске.

**Интерпретация:** большая доля помогает выбрать кандидатов для пилота с широким охватом имеющегося набора.

**Ограничение:** число записей не измеряет спрос и не подтверждает точность прогноза. Если это требуется для решения, нужны дополнительные данные и отдельная проверка модели.

### 10.5. График 2 — почему одного ценового ориентира мало

Сравним медианы и Q1–Q3 пяти марок. Точка — медиана, отрезок — центральные 50% цен. Это **разброс наблюдений**, а не доверительный интервал медианы и не интервал прогноза. В подписи каждой группы укажем n.

Различия марок могут отражать разный состав моделей, возраст и комплектацию. График не доказывает причинного влияния марки.

In [ ]:
story_prices = story_df[story_df['Make'].isin(story_top_makes)].groupby('Make').agg(
    n=('Price', 'size'), median=('Price', 'median'),
    q1=('Price', lambda values: values.quantile(.25)),
    q3=('Price', lambda values: values.quantile(.75))
).reindex(story_top_makes)
min_median = float(story_prices['median'].min())
max_median = float(story_prices['median'].max())
fig, ax = plt.subplots(figsize=(10, 5))
for y, (make, row) in enumerate(story_prices.iterrows()):
    ax.plot([row['q1'], row['q3']], [y, y], color='#b7c0c8', linewidth=8, solid_capstyle='round')
    ax.scatter(row['median'], y, color='#287a9e', s=65, zorder=3)
    ax.text(row['q3'] + 150, y, f'n={int(row["n"]):,}', va='center', fontsize=10)
ax.set_yticks(range(len(story_prices)), story_prices.index)
ax.invert_yaxis()
ax.set_xlim(0, float(story_prices['q3'].max()) * 1.3 + 500)
ax.set_xlabel('Цена предложения, евро; точка — медиана, отрезок — Q1–Q3')
ax.set_title(f'Медианы топ-групп: от {min_median:,.0f} до {max_median:,.0f} евро', loc='left', pad=18)
ax.grid(axis='y', visible=False)
fig.text(.02, .01, 'Разный состав моделей и годов выпуска может объяснять различия. Q1–Q3 не является интервалом прогноза.', fontsize=9)
fig.tight_layout(rect=(0, .06, 1, 1))
fig.savefig(story_dir / '02_price_ranges.png', dpi=140, bbox_inches='tight')
plt.show()
display(story_prices.round(1))

**Наблюдение:** укажите реальные медианы и Q1–Q3 двух выбранных марок.

**Интерпретация:** цены неоднородны; общий ориентир по всем автомобилям скрывает состав сегментов. Для дальнейшей проверки нужно уточнять модель и годы выпуска.

**Ограничение:** широкий диапазон не означает ошибки данных или плохой прогноз. Прогноз ещё не построен.

### 10.6. График 3 — где нужен осторожный охват

Массовая марка может содержать как хорошо представленные, так и редкие модели. Для каждой записи определим, сколько наблюдений имеет её группа марка–модель. Покажем долю **записей**, относящихся к малым группам. Это отличается от доли самих групп.

На одном графике показываем все записи. Не выделяем красным малые группы как «ошибки»: их небольшой размер — ограничение доступных данных.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.6))
large_n = story_total - story_small_n
large_share = 100 * large_n / story_total
ax.barh(['Все записи'], large_share, color='#287a9e', label=f'В группе n ≥ {STORY_MIN_N}')
ax.barh(['Все записи'], story_small_share, left=large_share, color='#b7c0c8', label=f'В группе n < {STORY_MIN_N}')
for left, width in [(0, large_share), (large_share, story_small_share)]:
    if width >= 8:
        ax.text(left + width/2, 0, f'{width:.1f}%', ha='center', va='center')
ax.set_xlim(0, 100)
ax.set_xlabel('Доля записей, %; группа = марка и модель')
ax.set_title(f'{story_small_share:.1f}% записей относятся к группам с n < {STORY_MIN_N}', loc='left', pad=18)
ax.legend(loc='upper center', bbox_to_anchor=(.5, -.25), ncol=2, frameon=False)
ax.grid(axis='y', visible=False)
fig.text(.02, .01, f'Учебный порог {STORY_MIN_N}: критерий планирования проверки, не гарантия точности.', fontsize=9)
fig.tight_layout(rect=(0, .05, 1, 1))
fig.savefig(story_dir / '03_segment_coverage.png', dpi=140, bbox_inches='tight')
plt.show()

story_sensitivity = pd.DataFrame([
    {'Порог n': threshold, 'Доля записей в группах ниже порога, %': 100*(story_model_n < threshold).mean()}
    for threshold in [10, 30, 50]
])
display(story_sensitivity.round(1))

**Наблюдение:** заполните долю записей в группах с n < 30 и сравните её с порогами 10 и 50.

**Интерпретация:** объём данных стоит проверять внутри сегментов, а не только на уровне марки. Размер групп может стать одним из критериев планирования пилота.

**Ограничение:** порог условный. После разбиения по году выпуска группы могут стать ещё меньше. Точность и пригодность модели всё равно нужно измерить на отложенных данных.

### 10.7. Собираем краткую историю из расчётов

Ниже — пример текста, который автоматически подставляет числа. Проверьте каждое предложение по таблицам. Вывод не должен зависеть от старых, вручную переписанных результатов.

Предлагаемое действие должно быть соразмерно доказательствам: здесь можно предложить **проверку пилота**, но нельзя рекомендовать закупку автомобилей, обещать рост выручки или заявлять, что модель уже точна.

In [ ]:
story_title = 'Пилот оценки цены: проверяем охват по сегментам'
story_context = 'Команда готовит инструмент оценки цены предложения и выбирает сегменты для первой проверки.'
story_facts = [
    f'В отчёте {story_total:,} записей; топ-{STORY_TOP_K} {STORY_MAKE_WORD} содержат {story_top_share:.1f}% этой базы.',
    f'Медианы цены топ-групп различаются от {min_median:,.0f} до {max_median:,.0f} евро; состав моделей и годов может объяснять различия.',
    f'{story_small_share:.1f}% записей относятся к группам {STORY_SEGMENT_NAME} с числом наблюдений меньше {STORY_MIN_N}.'
]
story_action = (
    'Следующий шаг: выбрать кандидатов для пилота по объёму записей, уточнить модель и годы выпуска, '
    'затем проверить ошибку и охват по сегментам на отложенных данных. Для малых групп обозначить ограничение '
    'или собрать дополнительные наблюдения до обещания одинакового качества.'
)
story_limit = (
    'Отчёт описывает подготовленный набор. Число записей не равно спросу; цены предложения не равны ценам сделок. '
    'Порог размера группы условный, а точность инструмента пока не проверена.'
)
print(story_title)
print(story_context)
for fact in story_facts:
    print('•', fact)
print(story_action)
print(story_limit)
story_prices.to_csv(story_dir / 'price_summary.csv')
story_sensitivity.to_csv(story_dir / 'threshold_sensitivity.csv', index=False)

### 10.8. Сохраняем историю для читателя без Python

Экспортируем последовательный HTML-отчёт с тремя статическими рисунками, наблюдениями, действием и ограничениями. Файл содержит изображения внутри себя и открывается без интернета. Отдельная интерактивная BI-панель остаётся инструментом исследования.

In [ ]:
import base64
import html

def story_image(filename):
    encoded = base64.b64encode((story_dir / filename).read_bytes()).decode('ascii')
    return 'data:image/png;base64,' + encoded

story_sections = []
for fact, filename in zip(story_facts, ['01_coverage.png', '02_price_ranges.png', '03_segment_coverage.png']):
    story_sections.append('<section><h2>' + html.escape(fact) + '</h2>'
                          + '<img alt="' + html.escape(fact, quote=True) + '" src="' + story_image(filename) + '"></section>')
report_html = ('<!doctype html><html lang="ru"><meta charset="utf-8">'
               '<meta name="viewport" content="width=device-width, initial-scale=1">'
               '<title>' + html.escape(story_title) + '</title>'
               '<style>body{max-width:960px;margin:36px auto;padding:0 20px;font:17px/1.55 system-ui;color:#22313c}'
               'h1{line-height:1.2}h2{font-size:21px}section{margin:36px 0}img{width:100%;height:auto}'
               '.note{padding:18px;background:#edf4f7;border-left:4px solid #287a9e}</style>'
               '<h1>' + html.escape(story_title) + '</h1><p>' + html.escape(story_context) + '</p>'
               '<p>Аудитория: руководитель команды сервиса объявлений. Источник: ' + html.escape(DATA_FILE)
               + '. Исключено по условиям истории: ' + str(int((~story_valid).sum())) + ' строк.</p>'
               + ''.join(story_sections)
               + '<div class="note"><h2>Предлагаемое действие</h2><p>' + html.escape(story_action)
               + '</p><h2>Ограничения</h2><p>' + html.escape(story_limit) + '</p></div></html>')
Path('cars_data_story.html').write_text(report_html, encoding='utf-8')
print('Готово: cars_data_story.html')

### 10.9. Редактируем заголовки, подписи и доказательства

| Было | Как улучшить |
|---|---|
| «График марок» | Указать главное рассчитанное наблюдение, его базу и долю |
| «Автоматы дороже» | Указать разницу медиан в конкретном сегменте, n и ограничения сравнения |
| «Нашли выбросы — удаляем» | Указать проверяемые записи, возможные причины и статус подтверждения |
| «Лучше начать с популярных марок» | Указать критерий объёма записей, кандидатов для проверки и ограничения охвата |

**Не подменяйте доказательства оформлением:** не скрывайте остальные категории; не меняйте шкалу между сравниваемыми графиками без объяснения; не показывайте только удобные сегменты; не называйте Q1–Q3 доверительным интервалом; не превращайте различие групп в причинный вывод.

**Редакторская проверка:** может ли читатель за минуту найти вопрос, главное наблюдение и следующий шаг? Понятен ли каждый знаменатель? Есть ли число и единица у каждого количественного утверждения? Не обещает ли текст больше, чем проверено?

### 10.10. Самостоятельное задание: своя Data Story

Выберите один вопрос: охват пилота; ценовые различия сравнимых сегментов; приоритет проверки качества данных. Аудиторию и предполагаемое решение задайте до отбора финальных графиков.

Подготовьте **одну страницу или три последовательных экрана**:

1. Контекст, вопрос и главная мысль в одном предложении.
2. Два или три графика с заголовками-наблюдениями. Для каждого — рассчитанное число, n, единица и короткая интерпретация.
3. Предлагаемое действие, ограничение и способ проверить результат следующего шага.

Добавьте в ноутбук небольшой журнал: «утверждение → таблица/расчёт → ограничение → действие». Если меняли фильтры, запишите условия и число исключённых строк. Сделайте выводы по своему запуску, не копируйте проценты из учебного примера.

**Что сдавать:** HTML-историю `cars_data_story.html`, ноутбук с воспроизводимыми расчётами и исходную BI-панель. Собственный сюжет и изменение оформления обязательны; автоматический экспорт примера без вашего анализа не является самостоятельной работой.

## 11. Самостоятельная работа: ответ заказчику

Подготовьте 4–6 графиков и короткий отчёт для команды сервиса объявлений.

1. Сформулируйте три бизнес-вопроса, на которые эти данные позволяют ответить.
2. Покажите структуру предложений в количествах и долях; объясните знаменатели.
3. Сравните цены двух или трёх сегментов: медиана, IQR, число наблюдений. Проверьте, меняется ли вывод после уточнения сегмента по модели и годам выпуска.
4. Найдите не менее пяти записей для проверки. Укажите `row_id`, визуальный признак проблемы, возможное объяснение и решение. Если подтвердить ошибку нельзя, оставьте статус проверки.
5. Покажите влияние предложенной фильтрации на размер выборки, среднюю и медианную цену. Отделите сценарий от подтверждённых исправлений.
6. Соберите компактную панель и сформулируйте три вывода: **наблюдение в числах → возможное действие → ограничение**.

Пример формы вывода (заполните своими числами): «На ___ марок приходится ___% записей; можно начать проверку модели на этих сегментах. Такой выбор не подтверждает спрос и не гарантирует перенос на редкие марки».

7. Подготовьте свою Data Story по разделу 10: один вопрос, 2–3 доказательства, действие и ограничения. Панель приложите к рассказу.

**Проверьте перед сдачей:** есть вопрос и единицы измерения; график не перегружен; масштаб не вводит в заблуждение; размер групп указан; заголовок соответствует данным; вывод опирается на рассчитанные значения.

**Оценивание, 10 баллов:** вопрос и показатели — 1; читаемость графиков — 2; сравнение сегментов — 1; проверка данных и обоснование решений — 2; BI-панель — 1; Data Story (связность, доказательства, действие и ограничения) — 3.

**Что сдавать:** выполненный ноутбук, HTML-панель, HTML-историю и CSV журнала проверки. Не добавляйте признаки, вычисленные из целевой цены, в будущую модель без отдельной схемы расчёта на обучающих данных.

Визуальный анализ связал показатели выборки с бизнес-вопросами и выделил записи для проверки. Подготовленный отчёт помогает выбрать сегменты для дальнейшего моделирования и объяснить ограничения будущей оценки цены.


In [ ]:
review.to_csv('cars_review_log.csv', index=False)
# Обоснованные исправления выполняйте в отдельной копии и описывайте в журнале.
# Исходные cars_processed.csv и cars_cleaned.csv здесь не перезаписываются.


## Документация

- [Matplotlib: Figure и Axes](https://matplotlib.org/stable/users/explain/quick_start.html)
- [Seaborn: сравнение категорий](https://seaborn.pydata.org/tutorial/categorical.html)
- [Seaborn: boxplot](https://seaborn.pydata.org/generated/seaborn.boxplot.html)
- [Plotly: treemap](https://plotly.com/python/treemaps/)
